# Geração de Relatório por Cliente

Este notebook é acionado via BeaZap (WhatsApp → Databricks) e gera um relatório em PDF para um cliente específico.

**Parâmetros de entrada:**
- `codigo_cliente` — Código do cliente extraído da mensagem WhatsApp
- `catalog` — Unity Catalog (padrão: `nazaria_dev`)
- `schema` — Schema do catálogo (padrão: `nazaria_gold`)
- `modo` — Modo do relatório: `cliente` ou `rede_agregado`
- `output_path` — Diretório de saída no DBFS (padrão: `/dbfs/FileStore/relatorios/`)

In [ ]:
# ── Widgets (parâmetros injetados pelo BeaZap via notebook_params) ────────────
dbutils.widgets.text("catalog",        "nazaria_dev",               "Catalog")
dbutils.widgets.text("schema",         "nazaria_gold",              "Schema")
dbutils.widgets.text("codigo_cliente", "",                          "Código Cliente")
dbutils.widgets.dropdown("modo",       "cliente", ["cliente", "rede_agregado"], "Modo")
dbutils.widgets.text("codigo_rede",    "",                          "Código Rede (se modo rede)")
dbutils.widgets.text("output_path",    "/dbfs/FileStore/relatorios/", "Caminho de saída")

In [ ]:
# ── Leitura dos parâmetros ────────────────────────────────────────────────────
catalog        = dbutils.widgets.get("catalog")
schema         = dbutils.widgets.get("schema")
codigo_cliente = dbutils.widgets.get("codigo_cliente").strip()
modo           = dbutils.widgets.get("modo")
codigo_rede    = dbutils.widgets.get("codigo_rede").strip()
output_path    = dbutils.widgets.get("output_path").rstrip("/") + "/"

# Validação básica
if not codigo_cliente:
    raise ValueError("Parâmetro 'codigo_cliente' não pode ser vazio.")

output_file = f"{output_path}relatorio_{codigo_cliente}.pdf"

print(f"Catalog      : {catalog}")
print(f"Schema       : {schema}")
print(f"Cód. Cliente : {codigo_cliente}")
print(f"Modo         : {modo}")
print(f"Output file  : {output_file}")

In [ ]:
# ── Instalação de dependências (apenas se necessário no cluster) ──────────────
# Descomente caso reportlab não esteja instalado no cluster
# %pip install reportlab -q

In [ ]:
# ── Consulta de dados do cliente ──────────────────────────────────────────────
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# ── Dados cadastrais ─────────────────────────────────────────────────────────
# Ajuste o nome das tabelas conforme o seu catálogo
df_cliente = spark.sql(f"""
    SELECT *
    FROM clientes
    WHERE codigo_cliente = '{codigo_cliente}'
    LIMIT 1
""")

if df_cliente.count() == 0:
    raise ValueError(f"Cliente '{codigo_cliente}' não encontrado em {catalog}.{schema}.clientes")

cliente_info = df_cliente.first().asDict()
print("Dados do cliente:", cliente_info)

In [ ]:
# ── Dados de consumo / métricas ───────────────────────────────────────────────
df_metricas = spark.sql(f"""
    SELECT
        data_referencia,
        valor_total,
        quantidade,
        descricao
    FROM metricas_cliente
    WHERE codigo_cliente = '{codigo_cliente}'
    ORDER BY data_referencia DESC
    LIMIT 12
""")

metricas_list = [row.asDict() for row in df_metricas.collect()]
print(f"Registros de métricas encontrados: {len(metricas_list)}")

In [ ]:
# ── Geração do PDF com ReportLab ──────────────────────────────────────────────
import os
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
)

# Garante que o diretório de saída existe
os.makedirs(output_path, exist_ok=True)

# ── Estilos ───────────────────────────────────────────────────────────────────
styles = getSampleStyleSheet()

style_title = ParagraphStyle(
    "Title",
    parent=styles["Heading1"],
    fontSize=20,
    textColor=colors.HexColor("#1a1a2e"),
    spaceAfter=6,
)
style_subtitle = ParagraphStyle(
    "Subtitle",
    parent=styles["Normal"],
    fontSize=11,
    textColor=colors.HexColor("#555555"),
    spaceAfter=12,
)
style_section = ParagraphStyle(
    "Section",
    parent=styles["Heading2"],
    fontSize=13,
    textColor=colors.HexColor("#16213e"),
    spaceBefore=16,
    spaceAfter=6,
)
style_body = styles["Normal"]

# ── Monta o documento ─────────────────────────────────────────────────────────
doc = SimpleDocTemplate(
    output_file,
    pagesize=A4,
    leftMargin=2 * cm,
    rightMargin=2 * cm,
    topMargin=2.5 * cm,
    bottomMargin=2 * cm,
)

story = []

# Cabeçalho
story.append(Paragraph("Relatório do Cliente", style_title))
story.append(Paragraph(
    f"Gerado em {datetime.now().strftime('%d/%m/%Y às %H:%M')} — Modo: {modo.upper()}",
    style_subtitle,
))
story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#cccccc")))
story.append(Spacer(1, 0.4 * cm))

# Dados cadastrais
story.append(Paragraph("Dados Cadastrais", style_section))

# Formata os dados do cliente em tabela 2 colunas
campos_cliente = [
    ["Código", codigo_cliente],
    ["Nome",   cliente_info.get("nome", "—")],
    ["E-mail", cliente_info.get("email", "—")],
    ["Cidade", cliente_info.get("cidade", "—")],
    ["Status", cliente_info.get("status", "—")],
]

table_cadastro = Table(campos_cliente, colWidths=[4 * cm, 12 * cm])
table_cadastro.setStyle(TableStyle([
    ("FONTNAME",    (0, 0), (0, -1), "Helvetica-Bold"),
    ("FONTNAME",    (1, 0), (1, -1), "Helvetica"),
    ("FONTSIZE",    (0, 0), (-1, -1), 10),
    ("TEXTCOLOR",   (0, 0), (0, -1), colors.HexColor("#333333")),
    ("ROWBACKGROUNDS", (0, 0), (-1, -1), [colors.HexColor("#f9f9f9"), colors.white]),
    ("GRID",        (0, 0), (-1, -1), 0.25, colors.HexColor("#e0e0e0")),
    ("TOPPADDING",  (0, 0), (-1, -1), 5),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ("LEFTPADDING", (0, 0), (-1, -1), 8),
]))

story.append(table_cadastro)
story.append(Spacer(1, 0.6 * cm))

# Métricas
story.append(Paragraph("Histórico de Métricas", style_section))

if metricas_list:
    header = [["Data", "Descrição", "Quantidade", "Valor Total"]]
    rows = [
        [
            str(r.get("data_referencia", ""))[:10],
            str(r.get("descricao", "")),
            str(r.get("quantidade", "")),
            f"R$ {float(r.get('valor_total', 0)):,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
        ]
        for r in metricas_list
    ]

    table_metricas = Table(
        header + rows,
        colWidths=[3 * cm, 8 * cm, 3 * cm, 3 * cm],
    )
    table_metricas.setStyle(TableStyle([
        ("BACKGROUND",  (0, 0), (-1, 0), colors.HexColor("#16213e")),
        ("TEXTCOLOR",   (0, 0), (-1, 0), colors.white),
        ("FONTNAME",    (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME",    (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE",    (0, 0), (-1, -1), 9),
        ("ALIGN",       (2, 0), (-1, -1), "RIGHT"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f4f4f4")]),
        ("GRID",        (0, 0), (-1, -1), 0.25, colors.HexColor("#cccccc")),
        ("TOPPADDING",  (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(table_metricas)
else:
    story.append(Paragraph("Nenhuma métrica encontrada para este cliente.", style_body))

# Rodapé
story.append(Spacer(1, 1 * cm))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#cccccc")))
story.append(Spacer(1, 0.2 * cm))
story.append(Paragraph(
    f"Gerado automaticamente pelo sistema BeaZap | {catalog}.{schema}",
    ParagraphStyle("Footer", parent=styles["Normal"], fontSize=8, textColor=colors.grey),
))

# Gera o PDF
doc.build(story)
print(f"PDF gerado com sucesso: {output_file}")

In [ ]:
# ── URL de download (FileStore) ───────────────────────────────────────────────
# Converte caminho DBFS para URL de download do workspace
download_path = output_file.replace("/dbfs/FileStore", "/files")
print(f"URL de download: {download_path}")

# Retorna o caminho do arquivo para o job run (capturado via task values)
dbutils.notebook.exit(output_file)